In [5]:
# Independent Model Trend Prediction

import numpy as np
import pyreadr
from tqdm import tqdm
import pandas as pd

# ============================================================
# Load cleaned snow data (same reorder as fitting stage)
# ============================================================
no_nbs = np.array([57,170,236,269,343,685,946,947,989,
                   1037,1084,1090,1109,1118,1127,1176,1203]) - 1

# Main data
snow_cleaned = pyreadr.read_r(
    r"D:\77\Research\temp\snow\snow_cleaned_full.Rda"
)
snow_cleaned = list(snow_cleaned.values())[0]

df1 = snow_cleaned.drop(index=no_nbs)
df2 = snow_cleaned.iloc[no_nbs]
all_y = pd.concat([df1, df2], axis=0).reset_index(drop=True)

y = all_y.iloc[:, 2:].to_numpy()      # S × TT
coords = all_y.iloc[:, :2].to_numpy() # S × 2

S, TT = y.shape
period = 52

print(f"S = {S}, TT = {TT}")

# ============================================================
# Load posterior samples (IND model p01 and p10)
# ============================================================
res01 = np.load(r"D:\77\Research\temp\snow\ind01.npz")
res10 = np.load(r"D:\77\Research\temp\snow\ind10.npz")

theta01 = res01["all_theta"]  # shape = (4S , M)
theta10 = res10["all_theta"]  # shape = (4S , M)

M = theta01.shape[1]
print(f"Loaded {M} posterior samples.")

# ============================================================
# Slice β blocks for IND model
# ============================================================
beta0_01 = theta01[0*S:1*S, :]     # block 1
beta1_01 = theta01[1*S:2*S, :]     # block 2
beta2_01 = theta01[2*S:3*S, :]     # block 3
alpha_01 = theta01[3*S:4*S, :]     # block 4

beta0_10 = theta10[0*S:1*S, :]
beta1_10 = theta10[1*S:2*S, :]
beta2_10 = theta10[2*S:3*S, :]
alpha_10 = theta10[3*S:4*S, :]

# ============================================================
# Allocate output arrays
# ============================================================
weekly_ini  = np.zeros((M, S, 52, 2))
weekly_final = np.zeros((M, S, 52, 2))

inv_logit = lambda x: 1 / (1 + np.exp(-x))

print("Begin computing transition probabilities...")

# ============================================================
# MAIN LOOP — identical to R code logic, but memory safe
# ============================================================
for m in tqdm(range(M), desc="Posterior samples"):

    # Extract one posterior sample
    b0  = beta0_01[:, m]
    b1  = beta1_01[:, m]
    b2  = beta2_01[:, m]
    a01 = alpha_01[:, m]

    b0s = beta0_10[:, m]
    b1s = beta1_10[:, m]
    b2s = beta2_10[:, m]
    a10 = alpha_10[:, m]

    # Initial state for each location
    init_state = np.column_stack([y[:,0] == 0, y[:,0] == 1]).astype(float)

    # -------------------------------------------------------------
    # Loop over 52 winter-week prediction targets
    # -------------------------------------------------------------
    for week_idx in range(1, 53):

        # ==========================================================
        # FIRST YEAR (same as R)
        # ==========================================================
        curr = init_state.copy()

        if week_idx > 1:
            for t in range(1, week_idx):
                p01 = inv_logit(b0 + b1*np.cos(2*np.pi*t/period)
                                    + b2*np.sin(2*np.pi*t/period)
                                    + a01*t)
                p10 = inv_logit(b0s + b1s*np.cos(2*np.pi*t/period)
                                      + b2s*np.sin(2*np.pi*t/period)
                                      + a10*t)

                c0 = curr[:,0]
                c1 = curr[:,1]
                curr = np.column_stack([
                    c0*(1-p01) + c1*p10,
                    c0*p01     + c1*(1-p10)
                ])

        weekly_ini[m,:,week_idx-1,:] = curr

        # ==========================================================
        # FINAL YEAR (52 years ahead)
        # ==========================================================
        curr = init_state.copy()
        final_step = week_idx + 51*52

        for t in range(1, final_step):
            p01 = inv_logit(b0 + b1*np.cos(2*np.pi*t/period)
                                + b2*np.sin(2*np.pi*t/period)
                                + a01*t)
            p10 = inv_logit(b0s + b1s*np.cos(2*np.pi*t/period)
                                  + b2s*np.sin(2*np.pi*t/period)
                                  + a10*t)

            c0 = curr[:,0]
            c1 = curr[:,1]
            curr = np.column_stack([
                c0*(1-p01) + c1*p10,
                c0*p01     + c1*(1-p10)
            ])

        weekly_final[m,:,week_idx-1,:] = curr

# ============================================================
# Save results
# ============================================================
np.savez_compressed(
    r"D:\77\Research\temp\snow\predict_ind.npz",
    weekly_ini=weekly_ini,
    weekly_final=weekly_final
)

print("Done! Saved to predict_ind.npz")


S = 1618, TT = 2704
Loaded 1000 posterior samples.
Begin computing transition probabilities...


Posterior samples: 100%|██████████| 1000/1000 [3:28:43<00:00, 12.52s/it] 


Done! Saved to predict_ind.npz


In [ ]:
# BYM Model Trend Prediction

import numpy as np
import pyreadr
from tqdm import tqdm
import pandas as pd

# ============================================================
# Load cleaned snow data (same reorder as fitting stage)
# ============================================================
no_nbs = np.array([57,170,236,269,343,685,946,947,989,
                   1037,1084,1090,1109,1118,1127,1176,1203]) - 1

snow_cleaned = pyreadr.read_r(
    r"D:\77\Research\temp\snow\snow_cleaned_full.Rda"
)
snow_cleaned = list(snow_cleaned.values())[0]

df1 = snow_cleaned.drop(index=no_nbs)
df2 = snow_cleaned.iloc[no_nbs]
all_y = pd.concat([df1, df2], axis=0).reset_index(drop=True)

y = all_y.iloc[:, 2:].to_numpy()
S, TT = y.shape
period = 52

print(f"S = {S}, TT = {TT}")

# ============================================================
# Load BYM posterior samples (CAR+IID)
# ============================================================
res01 = np.load(r"D:\77\Research\temp\snow\bym01.npz")
res10 = np.load(r"D:\77\Research\temp\snow\bym10.npz")

theta01 = res01["all_theta"]   # shape = (8S , M)
theta10 = res10["all_theta"]   # shape = (8S , M)

M = theta01.shape[1]
print(f"Loaded {M} posterior samples for BYM.")

# ============================================================
# Extract CAR + IID blocks for 4 covariates
# ============================================================
def split_bym_blocks(theta):
    """
    Input: 8S × M matrix
    Output: dict of β_j,CAR, β_j,IID each shape S×M
    """
    return {
        "b0_car": theta[0*S:1*S, :],
        "b0_iid": theta[1*S:2*S, :],
        "b1_car": theta[2*S:3*S, :],
        "b1_iid": theta[3*S:4*S, :],
        "b2_car": theta[4*S:5*S, :],
        "b2_iid": theta[5*S:6*S, :],
        "a_car" : theta[6*S:7*S, :],
        "a_iid" : theta[7*S:8*S, :]
    }

b01 = split_bym_blocks(theta01)
b10 = split_bym_blocks(theta10)

# Combine CAR + IID
def combine(beta):
    """Combine CAR + IID for each covariate."""
    return {
        "b0": beta["b0_car"] + beta["b0_iid"],
        "b1": beta["b1_car"] + beta["b1_iid"],
        "b2": beta["b2_car"] + beta["b2_iid"],
        "a" : beta["a_car"]  + beta["a_iid"],
    }

coef01 = combine(b01)
coef10 = combine(b10)

# ============================================================
# Allocate output arrays
# ============================================================
weekly_ini  = np.zeros((M, S, 52, 2))
weekly_final = np.zeros((M, S, 52, 2))

inv_logit = lambda x: 1 / (1 + np.exp(-x))

# ============================================================
# MAIN LOOP — identical to R logic but memory-safe
# ============================================================
print("Begin BYM prediction...")

for m in tqdm(range(M), desc="Posterior samples"):

    # Extract per-sample coefficients
    b0  = coef01["b0"][:, m]
    b1  = coef01["b1"][:, m]
    b2  = coef01["b2"][:, m]
    a01 = coef01["a"][:, m]

    b0s = coef10["b0"][:, m]
    b1s = coef10["b1"][:, m]
    b2s = coef10["b2"][:, m]
    a10 = coef10["a"][:, m]

    init_state = np.column_stack([y[:,0] == 0, y[:,0] == 1]).astype(float)

    for week_idx in range(1, 53):

        # -----------------------------------------------------
        # First year prediction
        # -----------------------------------------------------
        curr = init_state.copy()

        if week_idx > 1:
            for t in range(1, week_idx):

                p01 = inv_logit(b0 + b1*np.cos(2*np.pi*t/period)
                                    + b2*np.sin(2*np.pi*t/period)
                                    + a01*t)

                p10 = inv_logit(b0s + b1s*np.cos(2*np.pi*t/period)
                                      + b2s*np.sin(2*np.pi*t/period)
                                      + a10*t)

                c0 = curr[:,0]
                c1 = curr[:,1]

                curr = np.column_stack([
                    c0*(1-p01) + c1*p10,
                    c0*p01     + c1*(1-p10)
                ])

        weekly_ini[m,:,week_idx-1,:] = curr

        # -----------------------------------------------------
        # Final year prediction
        # -----------------------------------------------------
        curr = init_state.copy()
        final_steps = week_idx + 51*52

        for t in range(1, final_steps):

            p01 = inv_logit(b0 + b1*np.cos(2*np.pi*t/period)
                                + b2*np.sin(2*np.pi*t/period)
                                + a01*t)

            p10 = inv_logit(b0s + b1s*np.cos(2*np.pi*t/period)
                                  + b2s*np.sin(2*np.pi*t/period)
                                  + a10*t)

            c0 = curr[:,0]
            c1 = curr[:,1]

            curr = np.column_stack([
                c0*(1-p01) + c1*p10,
                c0*p01     + c1*(1-p10)
            ])

        weekly_final[m,:,week_idx-1,:] = curr


# ============================================================
# Save results
# ============================================================
np.savez_compressed(
    r"D:\77\Research\temp\snow\predict_bym.npz",
    weekly_ini=weekly_ini,
    weekly_final=weekly_final
)

print("BYM prediction completed and saved to predict_bym.npz!")


In [3]:
# ============================================================
#   BYM++ Trend Prediction (GPU Acceleration, batch = 10)
#   - Correct temp scaling (p01 / p10 separately)
#   - Continuous propagation for final year
# ============================================================

import numpy as np
import pyreadr
import pandas as pd
import torch
from tqdm import tqdm

# ============================================================
# Device
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ============================================================
# Load Snow Data
# ============================================================

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

snow_cleaned_full = pyreadr.read_r(
    r"D:\77\Research\temp\snow\snow_cleaned_full.Rda"
)
snow_cleaned_full = list(snow_cleaned_full.values())[0]

df1 = snow_cleaned_full.drop(index=no_nbs)
df2 = snow_cleaned_full.iloc[no_nbs]
all_y = pd.concat([df1, df2], axis=0).reset_index(drop=True).to_numpy()

y = all_y[:, 2:]            # shape (S, TT)
coords = all_y[:, :2]
S, TT = y.shape
period = 52

# ============================================================
# Load Temperature (aligned exactly as in MCMC)
# ============================================================

snow_temp_full = pyreadr.read_r(
    r"D:\77\Research\temp\snow\snow_temp_full.Rda"
)
snow_temp_full = list(snow_temp_full.values())[0].reset_index(drop=True)

temp = snow_temp_full.iloc[:, 2:]
temp1 = temp.drop(index=no_nbs)
temp2 = temp.iloc[no_nbs]
temp_aligned = pd.concat([temp1, temp2], axis=0)\
                  .reset_index(drop=True).to_numpy()   # (S, TT)

# ============================================================
# Elevation / Latitude scaling (match R::scale)
# ============================================================

curr_elev = pd.read_csv(
    r"D:\77\Research\temp\snow\curr_elev.csv"
).iloc[:, 3].to_numpy()

nnbs = pd.read_csv(
    r"D:\77\Research\temp\snow\nnbs_elev.csv", sep="\t"
).dropna(axis=1, how="all")

nnbs_elev = nnbs.iloc[:, -1].astype(float).to_numpy()

elev_raw = np.concatenate([curr_elev, nnbs_elev])
elev_scaled = (elev_raw - elev_raw.mean()) / elev_raw.std(ddof=1)

lat_raw = coords[:, 1]
lat_scaled = (lat_raw - lat_raw.mean()) / lat_raw.std(ddof=1)

# ============================================================
# TEMP scaling — EXACTLY as in MCMC
#   p01: Y_t = 0
#   p10: Y_t = 1
# ============================================================

# ---- p01 (0 -> 1) ----
loc01 = np.where(y[:, :-1] == 0)
pairs01 = np.column_stack(loc01)
pairs01 = pairs01[np.lexsort((pairs01[:,0], pairs01[:,1]))]
row01, time01 = pairs01[:,0], pairs01[:,1]

temp01_raw = temp_aligned[row01, time01]
temp01_mean = temp01_raw.mean()
temp01_sd   = temp01_raw.std()     # ddof=0 (same as MCMC)

# ---- p10 (1 -> 0) ----
loc10 = np.where(y[:, :-1] == 1)
pairs10 = np.column_stack(loc10)
pairs10 = pairs10[np.lexsort((pairs10[:,0], pairs10[:,1]))]
row10, time10 = pairs10[:,0], pairs10[:,1]

temp10_raw = temp_aligned[row10, time10]
temp10_mean = temp10_raw.mean()
temp10_sd   = temp10_raw.std()

print("TEMP scale p01: mean =", temp01_mean, "sd =", temp01_sd)
print("TEMP scale p10: mean =", temp10_mean, "sd =", temp10_sd)

# Full matrices for prediction use
temp01_scaled = (temp_aligned - temp01_mean) / temp01_sd
temp10_scaled = (temp_aligned - temp10_mean) / temp10_sd

# ============================================================
# Load BYM++ posterior samples
# ============================================================

res01 = np.load(r"D:\77\Research\temp\snow\bym++01.npz")
res10 = np.load(r"D:\77\Research\temp\snow\bym++10.npz")

theta01 = res01["all_theta"]   # (10*S + 2, M)
theta10 = res10["all_theta"]

M = theta01.shape[1]
print("Posterior samples M =", M)

# ============================================================
# Extract blocks
# ============================================================

def split(theta):
    b = {}
    for k in range(10):
        b[k] = theta[k*S:(k+1)*S, :]
    b["elev"] = theta[10*S, :]
    b["lat"]  = theta[10*S + 1, :]
    return b

β01 = split(theta01)
β10 = split(theta10)

def combine(b):
    return {
        "b0":   b[0] + b[1],
        "b1":   b[2] + b[3],
        "b2":   b[4] + b[5],
        "a":    b[6] + b[7],
        "temp": b[8] + b[9],
        "elev": b["elev"],
        "lat":  b["lat"]
    }

B01 = combine(β01)
B10 = combine(β10)

# ============================================================
# Allocate GPU tensors for constants
# ============================================================

elev_t  = torch.tensor(elev_scaled, device=device, dtype=torch.float32)
lat_t   = torch.tensor(lat_scaled,  device=device, dtype=torch.float32)
y0_t    = torch.tensor(y[:, 0],      device=device, dtype=torch.float32)

temp01_t = torch.tensor(temp01_scaled, device=device, dtype=torch.float32)
temp10_t = torch.tensor(temp10_scaled, device=device, dtype=torch.float32)

# ============================================================
# Output arrays (CPU)
# ============================================================

weekly_ini   = np.zeros((M, S, 52, 2), dtype=np.float32)
weekly_final = np.zeros((M, S, 52, 2), dtype=np.float32)

# ============================================================
# Transition helper
# ============================================================

def step_transition(p, p01, p10):
    p0 = p[:, :, 0]
    p1 = p[:, :, 1]
    new0 = p0*(1-p01) + p1*p10
    new1 = p0*p01     + p1*(1-p10)
    return torch.stack([new0, new1], dim=2)

inv_logit = torch.sigmoid
BATCH = 10

print("Begin GPU prediction (batch size = 10)...")

# ============================================================
# Main loop
# ============================================================

for start in tqdm(range(0, M, BATCH), desc="Posterior batches"):
    end = min(start + BATCH, M)
    m_batch = end - start

    # ---- load parameters ----
    b0   = torch.tensor(B01["b0"][:, start:end], device=device)
    b1   = torch.tensor(B01["b1"][:, start:end], device=device)
    b2   = torch.tensor(B01["b2"][:, start:end], device=device)
    a01  = torch.tensor(B01["a"][:,  start:end], device=device)
    t01  = torch.tensor(B01["temp"][:, start:end], device=device)
    eb   = torch.tensor(B01["elev"][start:end], device=device)
    lb   = torch.tensor(B01["lat"][ start:end], device=device)

    b0s  = torch.tensor(B10["b0"][:, start:end], device=device)
    b1s  = torch.tensor(B10["b1"][:, start:end], device=device)
    b2s  = torch.tensor(B10["b2"][:, start:end], device=device)
    a10  = torch.tensor(B10["a"][:,  start:end], device=device)
    t10  = torch.tensor(B10["temp"][:, start:end], device=device)
    ebs  = torch.tensor(B10["elev"][start:end], device=device)
    lbs  = torch.tensor(B10["lat"][ start:end], device=device)

    # ========================================================
    # FIRST YEAR (52 weeks, continuous)
    # ========================================================

    p = torch.stack([1 - y0_t, y0_t], dim=1)\
            .repeat(1, m_batch).reshape(S, m_batch, 2)

    for w in range(52):
        t = w + 1
        cos_t = np.cos(2*np.pi*t/period)
        sin_t = np.sin(2*np.pi*t/period)

        temp01_now = temp01_t[:, t-1]
        temp10_now = temp10_t[:, t-1]

        eta01 = (
            b0 + b1*cos_t + b2*sin_t + a01*t +
            t01 * temp01_now[:, None] +
            elev_t[:,None]*eb[None,:] +
            lat_t[:,None] *lb[None,:]
        )

        eta10 = (
            b0s + b1s*cos_t + b2s*sin_t + a10*t +
            t10 * temp10_now[:, None] +
            elev_t[:,None]*ebs[None,:] +
            lat_t[:,None] *lbs[None,:]
        )

        p = step_transition(p, inv_logit(eta01), inv_logit(eta10))
        weekly_ini[start:end, :, w, :] = p.permute(1,0,2).cpu().numpy()

    # ========================================================
    # FINAL YEAR (continuous from end of year 51)
    # ========================================================

    offset = 51 * 52

    p = torch.stack([1 - y0_t, y0_t], dim=1)\
            .repeat(1, m_batch).reshape(S, m_batch, 2)

    for t in range(1, offset + 1):
        cos_t = np.cos(2*np.pi*t/period)
        sin_t = np.sin(2*np.pi*t/period)

        temp01_now = temp01_t[:, t-1]
        temp10_now = temp10_t[:, t-1]

        eta01 = (
            b0 + b1*cos_t + b2*sin_t + a01*t +
            t01 * temp01_now[:, None] +
            elev_t[:,None]*eb[None,:] +
            lat_t[:,None] *lb[None,:]
        )

        eta10 = (
            b0s + b1s*cos_t + b2s*sin_t + a10*t +
            t10 * temp10_now[:, None] +
            elev_t[:,None]*ebs[None,:] +
            lat_t[:,None] *lbs[None,:]
        )

        p = step_transition(p, inv_logit(eta01), inv_logit(eta10))

    for w in range(52):
        t = offset + w + 1
        cos_t = np.cos(2*np.pi*t/period)
        sin_t = np.sin(2*np.pi*t/period)

        temp01_now = temp01_t[:, t-1]
        temp10_now = temp10_t[:, t-1]

        eta01 = (
            b0 + b1*cos_t + b2*sin_t + a01*t +
            t01 * temp01_now[:, None] +
            elev_t[:,None]*eb[None,:] +
            lat_t[:,None] *lb[None,:]
        )

        eta10 = (
            b0s + b1s*cos_t + b2s*sin_t + a10*t +
            t10 * temp10_now[:, None] +
            elev_t[:,None]*ebs[None,:] +
            lat_t[:,None] *lbs[None,:]
        )

        p = step_transition(p, inv_logit(eta01), inv_logit(eta10))
        weekly_final[start:end, :, w, :] = p.permute(1,0,2).cpu().numpy()

# ============================================================
# Save
# ============================================================

np.savez_compressed(
    r"D:\77\Research\temp\snow\predict_bympp.npz",
    weekly_ini=weekly_ini,
    weekly_final=weekly_final
)

print("Saved to predict_bympp.npz (GPU version). Done!")


Using device: cuda
TEMP scale p01: mean = 284.02623700347283 sd = 9.257789775966236
TEMP scale p10: mean = 261.2457462944966 sd = 11.299666607621113
Posterior samples M = 1000
Begin GPU prediction (batch size = 10)...


Posterior batches: 100%|██████████| 100/100 [03:42<00:00,  2.22s/it]


Saved to predict_bympp.npz (GPU version). Done!


In [4]:
import numpy as np
from pathlib import Path

import rpy2.robjects as ro
from rpy2.robjects import numpy2ri
from rpy2.robjects.conversion import localconverter


def predict_npz_to_rda(npz_path):
    """
    Convert predict_*.npz -> predict_*.Rda
    Keeps variable names exactly: weekly_ini, weekly_final
    Compatible with new rpy2 API (no deprecated activate()).
    """
    npz_path = Path(npz_path)
    rda_path = npz_path.with_suffix(".Rda")

    data = np.load(npz_path, allow_pickle=True)

    expected_keys = {"weekly_ini", "weekly_final"}
    if set(data.files) != expected_keys:
        raise ValueError(
            f"{npz_path.name} contains {data.files}, expected {expected_keys}"
        )


    with localconverter(ro.default_converter + numpy2ri.converter):
        for k in data.files:
            obj = data[k]
            if obj.dtype == object:
                obj = obj.tolist()
            ro.globalenv[k] = obj

    ro.r(f'save(weekly_ini, weekly_final, file="{rda_path.as_posix()}")')

    print(f"Saved {rda_path}")


base = r"D:/77/Research/temp/snow"

# predict_npz_to_rda(f"{base}/predict_ind.npz")
# predict_npz_to_rda(f"{base}/predict_bym.npz")
predict_npz_to_rda(f"{base}/predict_bympp.npz")



Saved D:\77\Research\temp\snow\predict_bympp.Rda
